# Interaction nets, live

*An interactive companion to Yves Lafont, "Interaction Nets", POPL 1990 ([doi:10.1145/96709.96718](https://doi.org/10.1145/96709.96718)).*

In 1990 Yves Lafont proposed a model of computation that is a kind of graph rewriting. In "interaction nets", a program
is a set of **agents** joined by wires, together with a handful of **rules** that say what
happens when two agents meet. There is no program counter, no global
memory, and no evaluation order to choose. Every step is local, so independent steps can
happen at the same time.

Lafont's original paper is an excellent exposition in the tradition of ["The Minimal Manual"](https://swcarpentry.github.io/swc-releases/2017.02/instructor-training/files/papers/carroll-minimal-manual-1987.pdf), so we will follow his pedagogy here.

This notebook follows Lafont's original paper section by section and can be read side by side with it. His paper's systems and examples appear here as nets you can run, draw and change.  Lafont calls inets "a new kind of programming language", a graphical one, and we follow suit.

We use the following Python packages:

* [`inets`](https://github.com/sigilante/inets)

See also these packages:

* [`vine-aviary`](https://github.com/sigilante/vine-aviary)
* [`inets-benchmarks`](https://github.com/sigilante/inets-benchmarks)

Run the cells in order (in JupyterLite, *Run ▸ Run All Cells*). Following Lafont's visual notation, an agent is an ellipse, and its **principal port** carries an arrowhead
pointing out along its wire. Two arrowheads tip to tip, ▼▲, mark two agents about to
interact.

In [2]:
try:
    import inets
except ImportError:          # in JupyterLite: fetch inets from PyPI
    %pip install -q "inets>=0.1.3,<0.2"
    import inets

import livenet as L
print("inets", inets.__version__)

inets 0.1.3


> To see every picture as a photocopy from 1990, run `L.retro(True)`; `L.retro(False)`
turns it off again.

In [3]:
L.retro(True)

## 1. Principles of interaction

* *[Lafont, pp. 95–100, Figures 1–5](https://dl.acm.org/doi/pdf/10.1145/96709.96718)*

First, some definitions:

* A **net** is an undirected graph with labeled vertices.
* Those vertices are **agents**.
* Each agent has a **symbol**, its name
(`Cons` or `Add`, say).
* Each symbol has a fixed number of **ports**.
* One port of each agent is its **principal port**; the others are **auxiliary**.
* A wire joins two ports. A port joined to nothing inside the net is a **free port**; free ports connect the net to the outside.

* Two agents joined by their principal ports form an **alive pair**.
* A **rule** rewrites an alive pair into a new net with the same free ports. Four things are true of rules:

    1. **Linearity.** Each free port of the pair appears exactly once in what the pair becomes. A rule never copies or drops anything by itself; agents must do any copying or erasing.
    2. **Binary interaction.** Agents interact only through their principal ports, and only in pairs.
    3. **No ambiguity.** Each pair of different symbols has at most one rule; no symbol has a rule for meeting itself.
    4. **No right-hand liveness.** A rule's right-hand side contains no alive pair, so no step leaves an interaction of its own undone.

### Lists

A list is commonly represented in as either an element and its continuation, or a terminator.  Here we will call those `x` for element; `u` for continuation; `Cons` for the pair of element and continuation; and `Nil` for the terminator.  We will have a rule to append to the list by replace the terminator with a pair of new element and terminator.

This table is in the `inets` format. The line `agent Cons + - -` declares `Cons` with three ports, principal port first. (Ignore the signs for now; section 2 explains them.)

In [4]:
lists = L.table("""
agent P      +
agent O      +
agent L      +
agent Cons   + - -       # Cons : list+ ; atom-, list-
agent Nil    +
agent Append - - +       # Append : list- ; list-, list+

rule Append (v w) >< Cons (x u)
  => u = Append (v t); w = Cons (x t)
  count append
rule Append (v w) >< Nil
  => w = v
  count append
""")

The first example appends two lists.

A list is either `Cons (x u)`, an element `x` followed by the rest `u`, or `Nil`. `Append` takes its first argument on its principal port; its auxiliary ports are the second argument and the result.

In [5]:
L.arrows(False)          # this figure predates principal ports: no arrowheads
agents = L.side_by_side(
    L.Net(lists, "_a = Cons (_b _c)"),
    L.Net(lists, "_a = Nil"),
    L.Net(lists, "Append (_b _c) = _a"),
    between="")          # no arrows between the pictures
L.arrows(True)
agents

Lafont draws a rule with the alive pair on the left, and what it becomes on the right. The free ports `v`, `w`, `x`, `u` are the same on both sides. The result `w` is on top; the
inputs hang below.

In [6]:
L.rule(lists, "Append (v w) = Cons (x u)", "u = Append (v t); w = Cons (x t)")

In [7]:
L.rule(lists, "Append (v w) = Nil", "w = v")

> We snuck something in there before defining it:  the arrowheads.  This enforces Rule #2:  each symbol has a **principal port** which is how it interacts.

In [8]:
agents = L.side_by_side(
    L.Net(lists, "_a = Cons (_b _c)"),
    L.Net(lists, "_a = Nil"),
    L.Net(lists, "Append (_b _c) = _a"))
agents

A net is written as equations between ports. `Append (v r) = Cons (P a1)` says that an
`Append` with auxiliary ports `v` and `r` has its principal port joined to that of a `Cons`.
A name used twice is a wire, and a name used once is a free port. Here is `[P, O] ++ [L]`,
reduced one interaction at a time (not a paper example):

In [9]:
n = L.Net(lists, "Append (v r) = Cons (P a1); a1 = Cons (O Nil); v = Cons (L Nil)")
n.steps()

In [10]:
print(n)

r = Cons (P Cons (O Cons (L Nil)))


The `Append` walks down the first list, one `Cons` per step, and meets the `Nil` at the end.
The time it takes grows with the first list, and it never looks at the second.

Unary maximum cannot be expressed that way, because two values need to be considered:

In [11]:
sym = L.table("""
agent Max 2
agent S 1
agent 0 0
""")   # no rules: for drawing only
L.arrows(False)
wrong = [L.rule(sym, "z = Max (0 y)", "z = y"),
         L.rule(sym, "z = Max (x 0)", "z = x"),
         L.rule(sym, "z = Max (S (x) S (y))", "z = S (Max (x y))")]
L.arrows(True)
L.side_by_side(*wrong, between="&nbsp;&nbsp;&nbsp;")

Thus, Rule #3 enforces no ambiguity, "there is *at most one* rule for each pair of distinct symbols $S,T$ and no rule for $S,S$.

See Lafont's Proposition 1 in this connection.

### Arithmetic: copying and erasing

Linearity means a rule cannot use anything twice or drop it. Multiplication has to do both:
`S x × y = x × y + y` uses `y` twice, and `0 × y = 0` drops it. So Lafont adds an agent
`Dupl` that copies a number and an agent `Erase` that destroys one (his figure 1). Numbers
are unary: `0`, `S (0)`, `S (S (0))`, and so on.

In [12]:
arith = L.table("""
agent 0     +
agent S     + -          # S : nat+ ; nat-
agent Add   - - +
agent Mult  - - +
agent Dupl  - + + duplicator   # Dupl : nat- ; {nat+, nat+}
agent Erase -
agent Max   - - +
agent Aux   - - +

rule Add (y z) >< 0
  => z = y
  count add
rule Add (y z) >< S (x)
  => x = Add (y t); z = S (t)
  count add
rule Mult (y z) >< 0
  => y = Erase; z = 0
  count mult
rule Mult (y z) >< S (x)
  => x = Mult (y1 w); w = Add (y2 z); y = Dupl (y1 y2)
  count mult
rule Dupl (a b) >< 0
  => a = 0; b = 0
  count dupl
rule Dupl (a b) >< S (x)
  => x = Dupl (p q); a = S (p); b = S (q)
  count dupl
rule Erase >< 0
  =>
  count erase
rule Erase >< S (x)
  => x = Erase
  count erase
rule Max (y z) >< 0
  => z = y
  count max
rule Max (y z) >< S (x)
  => y = Aux (x z)
  count max
rule Aux (x z) >< 0
  => z = S (x)
  count aux
rule Aux (x z) >< S (y)
  => x = Max (y t); z = S (t)
  count aux
""")

The rule for `S x × y`. The multiplication meets a successor and becomes three agents: a
multiplication by `x`, an addition, and a `Dupl` that copies `y` for the two of them.

In [13]:
L.rule(arith, "Mult (y z) = S (x)", "x = Mult (y1 w); w = Add (y2 z); y = Dupl (y1 y2)")

In [14]:
L.rule(arith, "Dupl (a b) = S (x)", "x = Dupl (p q); a = S (p); b = S (q)")

Now 2 × 2. The `Dupl` agents walk down the number they copy, and an `Erase` at the end eats
the copy nobody needs. First the opening steps, then the rest:

In [15]:
n = L.Net(arith, "Mult (two z) = S (S (0)); two = S (S (0))")
n.steps(limit=6)

In [11]:
n.reduce()
n

normal: 10 interactions (add 3, dupl 4, erase 3); 0 alive pairs left; 16 in all


Net('z = S (S (S (S (0))))')

### Order does not matter

A rule touches only its own pair, and two alive pairs never share an agent, so rules never
interfere. Lafont's **Proposition 1** (strong confluence) says that if a net can take two
different steps, each result can take one more step to reach the same net. It follows that
for a net that finishes, the order of reduction changes neither the result nor the number
of steps.

The 2 × 2 net above has up to three alive pairs at once. inets can reduce the newest pair
first (`stack`), the oldest first (`queue`), or pairs at random:

In [16]:
text = "Mult (two z) = S (S (0)); two = S (S (0))"
results = []
for order, seed in [("stack", 0), ("queue", 0), ("random", 1), ("random", 2), ("random", 3)]:
    n = L.Net(arith, text)
    print(order, seed, end=": ")
    n.reduce(order, seed=seed)
    results.append(n)
print("all the same:", all(results[0].same(m) for m in results))

stack 0: normal: 16 interactions (add 4, dupl 6, erase 3, mult 3); 0 alive pairs left; 16 in all
queue 0: normal: 16 interactions (add 4, dupl 6, erase 3, mult 3); 0 alive pairs left; 16 in all
random 1: normal: 16 interactions (add 4, dupl 6, erase 3, mult 3); 0 alive pairs left; 16 in all
random 2: normal: 16 interactions (add 4, dupl 6, erase 3, mult 3); 0 alive pairs left; 16 in all
random 3: normal: 16 interactions (add 4, dupl 6, erase 3, mult 3); 0 alive pairs left; 16 in all
all the same: True


### Maximum needs a helper

Some functions cannot be written with one agent. The maximum of two numbers would have to
look at both arguments at once, but an agent sees only what is on its principal port. So
`Max` looks at its first argument, and if that is a successor it hands over to a second
agent, `Aux`, which looks at the other argument (figure 2). The program has to choose which
argument to look at first. Lafont notes that this rules out functions like "parallel or",
which must answer as soon as either argument does.

In [17]:
L.rule(arith, "Max (y z) = S (x)", "y = Aux (x z)")

In [18]:
L.rule(arith, "Aux (x z) = S (y)", "x = Max (y t); z = S (t)")

In [19]:
n = L.Net(arith, "Max (one z) = S (S (0)); one = S (0)")
n.steps()

Try changing the numbers: `Max (two z) = S (0); two = S (S (0))` is max(1, 2).


In [20]:
n = L.Net(arith, "Max (two z) = S (0); two = S (S (0))")
n.steps()

### A computation that never ends

Lafont finally adds Rule #4: a rule's right-hand side contains no alive pair, so
no step leaves an interaction of its own undone. That still does not make every computation finite. His **turnstile** (figure 3) has one rule, `Turn` meeting `Stile`, and a net of one `Turn` and two `Stile`s comes out of that rule unchanged.

The paper gives these two agents no types. They can be given some, though: `Stile` an
output, and `Turn` an input on both its ports. We use those signs here for the `inets` package.

In [21]:
turnstile = L.table("""
agent Turn  - -
agent Stile +

rule Turn (x) >< Stile
  => x = Turn (s); s = Stile
  count turn
""")
turnstile.check()

[]

In [22]:
n = L.Net(turnstile, "Turn (s) = Stile; s = Stile")
n.steps(limit=4)

In [23]:
n.reduce(limit=1000)
print(n.same("Turn (s) = Stile; s = Stile"))

interactions: 1000 interactions (turn 1000); 1 alive pairs left; 1004 in all
True


### Concatenation in constant time

Appending lists takes time that grows with the first list. Logic programmers use a trick: a
**difference-list** keeps a pointer to the hole at the end of the list, so you can plug
another list straight in rather than traversing every single time. In a net that pointer is just a wire. `Diff (x y)` holds the front of a list, `x`, and the hole at its end, `y`. `D_append` plugs one difference-list into
another in two steps, however long the lists are (Figure 4).

> Here you will see our automated layout diverge from Lafont's.  Satisfy yourself that the same rules are being followed.

In [24]:
difflists = L.table("""
agent P        +
agent O        +
agent L        +
agent Cons     + - -
agent Nil      +
agent Diff     + - +             # Diff : d_list+ ; {list-, list+}
agent D_append - - +             # D_append : d_list- ; d_list-, d_list+
agent Open     - - +             # Open : d_list- ; list-, list+

rule D_append (v w) >< Diff (x y)
  => w = Diff (x t); v = Open (t y)
  count d_append
rule Open (a b) >< Diff (x y)
  => b = x; a = y
  count open
""")

In [25]:
L.rule(difflists, "D_append (v w) = Diff (x y)", "w = Diff (x t); v = Open (t y)")

Lafont illustrates `POP ++ L`. In the first list the last tail, `b`, is the hole, joined back to its `Diff`; in the second list the hole is `e`.

In [26]:
n = L.Net(difflists, "D_append (d r) = Diff (a b); a = Cons (P a1); a1 = Cons (O a2); "
                     "a2 = Cons (P b); d = Diff (c e); c = Cons (L e)")
n.steps()

In [27]:
print(n)

r = Diff (Cons (P Cons (O Cons (P Cons (L w1)))) w1)


### Parsing

His second example (p. 100 and Figure 5) reads arithmetic written in Polish (prefix) notation, such as `+ 1 1`, from a stream of symbols, and builds a tree (figure 5). The stream is a chain of agents, one per symbol. `Parse` eats one symbol at a time. At a `+` it splits into two parsers, one for each operand; the second waits for whatever stream the first leaves behind.

> `inets`'s table reader does not accept `+` and `1` as agent names, so here the stream's symbols are `plus` and `one`, and the tree is built of `Plus` and `One`.

In [30]:
parse = L.table("""
agent plus  + -          # + : stream+ ; stream-
agent one   + -          # 1 : stream+ ; stream-
agent Plus  + - -        # Plus : tree+ ; tree-, tree-
agent One   +
agent Parse - + +        # Parse : stream- ; {tree+, stream+}

rule Parse (u y) >< plus (x)
  => u = Plus (v w); x = Parse (v s); s = Parse (w y)
  count parse
rule Parse (u y) >< one (x)
  => u = One; y = x
  count parse
""")

In [31]:
n = L.Net(parse, "Parse (u y) = plus (s1); s1 = one (s2); s2 = one (x)")
n.steps()

`u` is now the tree `Plus (One One)`, and `y = x`: the rest of the stream, never read, is
passed on. Try `+ + 1 1 1`:
`Parse (u y) = plus (s1); s1 = plus (s2); s2 = one (s3); s3 = one (s4); s4 = one (x)`.

### How much can nets compute?

Lafont remarks that Turing machines and $SK$ combinators are easily simulated. The `inets` one such table of its own for the combinators $S$, $K$ and $I$, as a natural extension to combinatory logic. `ski.run`
takes a term in prefix form, where a backtick applies the term after it to the one after
that, so ` ```SKKI ` is `((S K) K) I`. It reduces lazily rather than eagerly and reports `wanted` when complete. `S K K I` is `I`:

In [34]:
from inets import ski
r = ski.run("```SKKI")
print(r.result.reason, r.result.interactions, "interactions:", ski.show(r.term))

wanted 6 interactions: I


## 2. A type discipline

* *[Lafont, pp. 100–104, and figures 6–7](https://dl.acm.org/doi/pdf/10.1145/96709.96718).*

Not every pair of symbols should have a rule: `Cons` and `Nil` have nothing to do with each
other. Lafont limits which agents may meet by giving their ports **types**.

Each port has a type and a direction: an **output** (`list+`) or an **input** (`list-`). A
net is well typed if every wire joins an output to an input of the same type. Two symbols
**match** if their principal ports have opposite directions. A symbol whose principal port
is an output, like `Cons`, is a **constructor**; one whose principal port is an input, like
`Append`, is a **destructor**. Which direction to call "input" is a convention, and nothing
in the theory favours constructors over destructors.

Lafont then asks for **completeness**: a rule for each pair of matching symbols. With it, a
well-typed alive pair can always be reduced.

5. **Typing**.  Rules are well-typed.
6. **Completeness**.  There is a rule for each pair of matching symbols.

The `+` and `-` in the tables above are these directions, principal port first. inets
checks a table against them, but it does not know the paper's type names (`list`, `atom`,
`nat`). So it reports `Append` meeting an atom as a missing rule, a meeting that the paper's
types rule out:

In [38]:
lists.check()

['no rule for Append >< L',
 'no rule for Append >< O',
 'no rule for Append >< P']

In [39]:
arith.check()

[]

### When a typed net stops

The paper's **Proposition 2** describes how a net can come to a stop. Take a well-typed, finite, nonempty net in which no rule applies. Start at any agent and follow principal ports, from agent to agent. Either you reach a free port, and the net waits for its surroundings, or you come back to where you started.

The second case is a **vicious circle** or de facto an infinite loop: a ring of agents, each with its principal port facing an auxiliary port of the next. No agent in it can ever interact. In Lafont's words (**Proposition 3**), "A vicious circle stays forever."

Vicious circles can appear during a computation. Feed `Append`'s result back into its own second argument, and append the list `P, O, P`:

In [40]:
n = L.Net(lists, "Append (k k) = Cons (P a1); a1 = Cons (O a2); a2 = Cons (P a3); a3 = Nil")
n.steps()

The list has become a ring. It is well typed and has no alive pair, and it never will. The
degenerate case is `Append (k k) = Nil`: one step leaves a loop of bare wire, which Lafont
draws as an empty circle. inets does not represent a wire with no agents, so here the net
becomes empty:

In [45]:
n = L.Net(lists, "Append (k k) = Nil")
n

Net('Append (w1 w1) = Nil')

In [46]:
n.reduce()
print(repr(str(n)))

normal: 1 interactions (append 1); 0 alive pairs left; 1 in all
''


### Simple nets

Forbidding cycles is not a sufficient answer because the difference-lists and the parser need them.  Lafont instead provides some heuristics to tell good cycles from bad ones. Each symbol's auxiliary ports are divided into groups, a **partition**. Most symbols put each port in a group of its own. `Dupl` puts its two copies in one group, and `Erase` has a single empty group (p. 102); `Diff` and `Parse` also put their two auxiliary ports together (figure 6).

A net is **simple** if it can be built using only three operations: *link*, a single wire;
*cut*, joining two simple nets by one wire; and *graft*, joining a new agent to simple nets,
one for each group of its partition. (See his diagrams of these operations on pp. 102–103.) Allowing two more operations, the empty net and
putting two nets side by side, gives the larger class of **semi-simple** nets. When every
port is in a group of its own, a simple net is a connected net with no cycle, and a semi-simple net is one with no cycle (Proposition 4).

Then:

- a semi-simple net contains no vicious circle (Proposition 5);
- if every rule is simple, reduction keeps a net simple (Proposition 6). Lafont describes this as essentially the cut-elimination theorem of multiplicative linear logic, adapted.

So a last condition asks that rules be simple, and then a simple net never deadlocks
(Proposition 7). With suitable partitions, every rule in the paper is simple.

The vicious circle came from a net that wasn't simple: `Append`'s two auxiliary ports are in different groups, yet one wire joined them.

> `inets` does not record partitions and does not check simplicity.

## 3. A programming language

* *[Lafont, pp. 104–106, and figures 8–11](https://dl.acm.org/doi/pdf/10.1145/96709.96718).*

Lafont sketches a concrete language for inets from three parts:
* A `type` declaration lists the types;
* A`symbol` declaration gives each symbol's port types (principal port first, partitions in
braces);
* and then the interaction rules.

In his notation the `Append` rule for `Cons` is

    Cons [x, Append(v, t)]  )(  Append [v, Cons(x, t)]

Each side of `)(` is one agent of the pair. Its brackets list what each of its auxiliary
ports is joined to after the interaction, and names are shared across the two sides. So the
`Cons`'s tail now feeds a new `Append (v, t)`, and the old `Append`'s result becomes a new
`Cons (x, t)`. In the inets format the same rule is

    rule Append (v w) >< Cons (x u)
      => u = Append (v t); w = Cons (x t)
      count append

where `count` names the rule in the tallies inets keeps.

He also says how to run such programs on a regular computer: agents are cells, wires are
pointers, and the alive pairs wait on a stack. Since no agent is ever shared, no garbage
collector is needed.

> `inets`'s default order, `stack`, does the same: it takes the newest
alive pair first.

For input and output, he suggests treating a keyboard as an endless stream of agents, one
per character, much like the parser's stream above.

A couple of pages of example programs in his notation follow (pp. 105–106), then he concludes.

### Further reading

* [Lafont, *Interaction Combinators*](https://www.sciencedirect.com/science/article/pii/S0890540197926432) (Information and Computation, 1997): three agents and six rules that can simulate any interaction net. inets bundles them as `inets.bundled("combinators")`.
* [`inets`](https://github.com/sigilante/inets)
* [`inets-benchmarks`](https://github.com/sigilante/inets-benchmarks), which includes among its growing set of examples all of the valid nets in this notebook
* [Xie, Yuheng, *iNet: Programming with interaction nets*](https://web.archive.org/web/20230907061237/https://inet.run/), a simple programming language for inets.
* [*The Vine Programming Language*](https://vine.dev/docs/), targeting a novel system architecture based on inets.